# 🧬 Chemical Entity Resolution & Catalog Deduplication Pipeline
**Objective:** Match unstructured, multi-vendor chemical catalog listings against a canonical Master Chemical Registry using TF-IDF N-gram Vectorization and Cosine Similarity.

---
### **Pipeline Workflow**
1. **Canonical Registry & Vendor Catalog Ingestion:** Load canonical master compounds alongside unstandardized supplier raw product titles.
2. **Text Normalization & N-gram Tokenization:** Preprocess chemical text and convert names into character n-grams.
3. **TF-IDF Vectorization & Similarity Matrix:** Compute pairwise cosine similarity scores to identify entity matches.
4. **Resolution Thresholding:** Assign match confidence flags (`HIGH_CONFIDENCE`, `NEEDS_REVIEW`, `NO_MATCH`).

In [1]:
import re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Display options
pd.set_option('display.max_colwidth', None)

print("✅ Machine Learning & NLP libraries imported successfully!")

✅ Machine Learning & NLP libraries imported successfully!


In [2]:
# 1. Canonical Master Database (Ground Truth)
master_registry = pd.DataFrame([
    {"master_id": "CHEM-001", "canonical_name": "Isopropanol", "cas_number": "67-63-0"},
    {"master_id": "CHEM-002", "canonical_name": "Acetone", "cas_number": "67-64-1"},
    {"master_id": "CHEM-003", "canonical_name": "Sodium Hydroxide", "cas_number": "1310-73-2"},
    {"master_id": "CHEM-004", "canonical_name": "Tetrahydrofuran", "cas_number": "109-99-9"},
    {"master_id": "CHEM-005", "canonical_name": "Dichloromethane", "cas_number": "75-09-2"}
])

# 2. Messy Vendor Listings (No CAS Numbers, raw messy titles, brand variations)
vendor_catalog = pd.DataFrame([
    {"vendor_sku": "SIG-8821", "vendor_name": "Sigma-Aldrich", "raw_title": "2-Propanol / Isopropyl Alcohol (IPA), ACS Reagent 99.5%"},
    {"vendor_sku": "FISH-1029", "vendor_name": "Thermo Fisher", "raw_title": "Acetone, HPLC Grade, >=99.9%"},
    {"vendor_sku": "VWR-4412", "vendor_name": "VWR Catalog", "raw_title": "Sodium Hydroxide Pellets, ACS Reagent Grade"},
    {"vendor_sku": "SIG-3301", "vendor_name": "Sigma-Aldrich", "raw_title": "THF (Tetrahydrofuran), Anhydrous, 99.9%"},
    {"vendor_sku": "VWR-9981", "vendor_name": "VWR Catalog", "raw_title": "Methylene Chloride (Dichloromethane), ACS Grade"},
    {"vendor_sku": "UNKNOWN-01", "vendor_name": "Custom Supplier", "raw_title": "Novel Polymeric Catalyst Matrix X-100"} # Intentional non-match
])

print(f"✅ Loaded {len(master_registry)} master registry entries.")
print(f"✅ Loaded {len(vendor_catalog)} unstandardized vendor items.")

✅ Loaded 5 master registry entries.
✅ Loaded 6 unstandardized vendor items.


In [3]:
def normalize_text(text: str) -> str:
    """Cleans punctuation, lowercases, and normalizes white space."""
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    return ' '.join(text.split())

# Apply normalization to canonical names and raw vendor titles
master_registry['clean_canonical'] = master_registry['canonical_name'].apply(normalize_text)
vendor_catalog['clean_title'] = vendor_catalog['raw_title'].apply(normalize_text)

# Initialize Character 3-Gram TF-IDF Vectorizer
# Character n-grams capture sub-string similarities (e.g., "tetrahydrofuran" vs "thf tetrahydrofuran")
vectorizer = TfidfVectorizer(analyzer='char', ngram_range=(3, 3))

# Fit vectorizer on master dataset and transform both master and vendor datasets
master_tfidf = vectorizer.fit_transform(master_registry['clean_canonical'])
vendor_tfidf = vectorizer.transform(vendor_catalog['clean_title'])

print(f"✅ TF-IDF Matrix created! Vocabulary size: {len(vectorizer.get_feature_names_out())} character 3-grams.")

✅ TF-IDF Matrix created! Vocabulary size: 51 character 3-grams.


In [4]:
# Compute pairwise Cosine Similarity (Vendor Matrix x Master Matrix)
similarity_matrix = cosine_similarity(vendor_tfidf, master_tfidf)

resolved_results = []

# Threshold definitions for entity matching
HIGH_CONF_THRESHOLD = 0.45
LOW_CONF_THRESHOLD = 0.20

for idx, vendor_row in vendor_catalog.iterrows():
    # Find highest similarity score for this vendor item against master registry
    sim_scores = similarity_matrix[idx]
    best_match_idx = np.argmax(sim_scores)
    best_score = sim_scores[best_match_idx]
    
    matched_master = master_registry.iloc[best_match_idx]
    
    # Classify match status based on similarity score thresholds
    if best_score >= HIGH_CONF_THRESHOLD:
        match_status = "MATCH_HIGH_CONFIDENCE"
    elif best_score >= LOW_CONF_THRESHOLD:
        match_status = "NEEDS_HUMAN_REVIEW"
    else:
        match_status = "NO_MATCH_FOUND"
        
    resolved_results.append({
        "vendor_sku": vendor_row["vendor_sku"],
        "raw_vendor_title": vendor_row["raw_title"],
        "matched_master_id": matched_master["master_id"] if match_status != "NO_MATCH_FOUND" else "N/A",
        "matched_canonical_name": matched_master["canonical_name"] if match_status != "NO_MATCH_FOUND" else "N/A",
        "matched_cas_number": matched_master["cas_number"] if match_status != "NO_MATCH_FOUND" else "N/A",
        "similarity_score": round(float(best_score), 4),
        "match_status": match_status
    })

df_resolved = pd.DataFrame(resolved_results)

# Display Entity Resolution Output
display(df_resolved[['vendor_sku', 'raw_vendor_title', 'matched_canonical_name', 'similarity_score', 'match_status']])

,vendor_sku,raw_vendor_title,matched_canonical_name,similarity_score,match_status
0,SIG-8821,"2-Propanol / Isopropyl Alcohol (IPA), ACS Reagent 99.5%",Isopropanol,0.9467,MATCH_HIGH_CONFIDENCE
1,FISH-1029,"Acetone, HPLC Grade, >=99.9%",Acetone,1.0000,MATCH_HIGH_CONFIDENCE
2,VWR-4412,"Sodium Hydroxide Pellets, ACS Reagent Grade",Sodium Hydroxide,1.0000,MATCH_HIGH_CONFIDENCE
3,SIG-3301,"THF (Tetrahydrofuran), Anhydrous, 99.9%",Tetrahydrofuran,0.9530,MATCH_HIGH_CONFIDENCE
4,VWR-9981,"Methylene Chloride (Dichloromethane), ACS Grade",Dichloromethane,0.9270,MATCH_HIGH_CONFIDENCE
5,UNKNOWN-01,Novel Polymeric Catalyst Matrix X-100,N/A,0.0000,NO_MATCH_FOUND


## **Entity Resolution Performance Analysis**

### **Why Character N-Gram Vectorization Outperforms Word Tokenization**
1. **Synonym & IUPAC Resiliency:** Traditional word tokenization fails when vendor names use parenthetical synonyms or chemical abbreviations (e.g., `Methylene Chloride (Dichloromethane)` vs. `Dichloromethane`).
2. **Sub-string Matching:** By breaking chemical strings into overlapping 3-character n-grams (`analyzer='char', ngram_range=(3,3)`), the TF-IDF vectorizer measures morphological similarity across names without requiring hardcoded dictionary maps.
3. **Automated Confidence Triage:**
   * **Score >= 0.45:** Automatically mapped to canonical master entity (`MATCH_HIGH_CONFIDENCE`).
   * **Score 0.20 – 0.44:** Routed to subject matter experts (`NEEDS_HUMAN_REVIEW`).
   * **Score < 0.20:** Excluded from auto-matching to prevent catalog corruption (`NO_MATCH_FOUND`).

In [6]:
# Save clean resolved entity map to CSV
output_resolved_csv = "resolved_chemical_catalog_map.csv"
df_resolved.to_csv(output_resolved_csv, index=False)

print(f"✅ Entity Resolution Pipeline Complete!")
print(f"📁 Clean mapping table saved to '{output_resolved_csv}'.")

✅ Entity Resolution Pipeline Complete!
📁 Clean mapping table saved to 'resolved_chemical_catalog_map.csv'.
